In [1]:
# Install dependencies
%pip install --upgrade pip 
%pip install pandas requests

Note: you may need to restart the kernel to use updated packages.
  Using cached pandas-2.3.1-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
  Using cached numpy-2.3.2-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached charset_normalizer-3.4.3-cp313-cp313-win_amd64.whl.metadata (37 kB)
  Using cached idna-3.10-py3-none-any.whl.metadata (10 kB)
  Using cached urllib3-2.5.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached certifi-2025.8.3-py3-none-any.whl.metadata (2.4 kB)
Using cached pandas-2.3.1-cp313-cp313-win_amd64.whl (11.0 MB)
Using cached requests-2.32.4-py3-none-any.whl (64 kB)
Using cached charset_normalizer-3.4.3-cp313-cp313-win_amd64.whl (107 kB)
Using cached idna-3.10-py3-none-any.whl (70 kB)
Using cached urllib3-2.5.0-py3-none-any.whl (129 kB)
Using cached certifi-2025.8.3-p

In [ ]:
import pandas as pd
import requests
import json
import os
import hashlib
import datetime
from urllib.parse import urlparse
import subprocess


In [ ]:
input_url = ["https://computo.oep.org.bo/", "https://sirepre.oep.org.bo/"]
data_dir = "../data"

In [ ]:
def ping(url):
    try:
        # Usar '-I' para obtener solo el encabezado y verificar la conexión
        result = subprocess.run(
            ["curl", "-I", url],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=5,
        )

        if result.returncode != 0:
            print(f"No se pudo alcanzar {url}. Saltando todo.")
            raise SystemExit
        else:
            print(f"{url} respondió correctamente.")
    except Exception as e:
        print(f"Error al hacer curl a {url}: {e}. Saltando todo.")
        return False
        #raise SystemExit


    return True

http://131.0.1.19:3002/ respondió correctamente.


In [ ]:
def save_data(json_data, filename):
    with open(os.path.join(data_dir, filename), "w") as f:
        json.dump(json_data, f, ensure_ascii=False, indent=4)
    print(f"Data saved to {filename}.")

In [ ]:
def extract_data(url, method="GET"):
    if not ping(url):
        return

    if method.upper() == "POST":
        response = requests.post(url, timeout=100)
    else:
        response = requests.get(url, timeout=100)

    if response.status_code in [200, 201]:
        data = response.json()
        # Get the URL path without the domain, remove leading/trailing '/'
        path = urlparse(url).path.strip("/")
        # Replace '/' with '_' to create a valid filename
        filename = path.replace("/", "_") + ".json"
        save_data(data, filename)

        metadata = {
            "source": url,
            "request_status": response.status_code,
            "timestamp": datetime.datetime.now().isoformat(),
            "unix_timestamp": int(datetime.datetime.now().timestamp()),
            "hash": hashlib.md5(json.dumps(data).encode("utf-8")).hexdigest(),
        }

        save_data(metadata, f"metadata-{filename}")
    else:
        print(f"Failed to extract data from {url}. Status code: {response.status_code}")

In [ ]:
for url in input_url:
    extract_data(url)